In [29]:
import duckdb
con = duckdb.connect("../data/power.duckdb")
con.sql("SET Timezone = 'UTC'")

SuccessBOOLEAN


In [30]:
con.sql("""
        CREATE OR REPLACE TABLE prices_raw AS
        SELECT *
        FROM read_parquet('../data/raw/prices.parquet')
""")

con.sql("SELECT * FROM prices_raw LIMIT 5").df() ##load the Parquet file into a table

,ts_utc,price_euro_mwh
0,2018-09-30 22:00:00+00:00,59.53
1,2018-09-30 23:00:00+00:00,56.10
2,2018-10-01 00:00:00+00:00,51.41
3,2018-10-01 01:00:00+00:00,47.38
4,2018-10-01 02:00:00+00:00,47.59


In [31]:
con.sql("""
    SELECT
        COUNT(*)            AS n_rows,
        COUNT(price_euro_mwh)      AS n_with_price,
        MIN(ts_utc)         AS first_hour,
        MAX(ts_utc)         AS last_hour
    FROM prices_raw
""").df() ##basic quality checks

,n_rows,n_with_price,first_hour,last_hour
0,70897,70801,2018-09-30 22:00:00+00:00,2026-11-01 22:00:00+00:00


In [32]:
con.sql("""
    SELECT ts_utc, count(*) AS n
    FROM prices_raw
    GROUP BY ts_utc
    HAVING COUNT(*) > 1
""").df() ##any duplicate hours?

,ts_utc,n


In [33]:
con.sql(""" 
    SELECT
        CAST(ts_utc AT TIME ZONE 'Europe/Berlin' AS DATE) AS day_local,
        COUNT(*) AS n_hours
    FROM prices_raw
    WHERE price_euro_mwh IS NOT NULL
    GROUP BY day_local
    HAVING COUNT(*) <> 24
    ORDER BY day_local
""").df()

,day_local,n_hours
0,2018-10-28,25
1,2019-03-31,23
2,2019-10-27,25
3,2020-03-29,23
4,2020-10-25,25
5,2021-03-28,23
6,2021-10-31,25
7,2022-03-27,23
8,2022-10-30,25
9,2023-03-26,23


In [34]:
con.sql(""" 
    CREATE OR REPLACE TABLE prices_clean AS
    WITH with_local AS (
        SELECT 
            ts_utc,
            ts_utc AT TIME ZONE 'Europe/Berlin' AS ts_local,
            price_euro_mwh
        FROM prices_raw
        WHERE price_euro_mwh IS NOT NULL
    )
    SELECT ts_utc,
        ts_local,
        price_euro_mwh,
        year(ts_local)   AS year,
        month(ts_local)  AS month,
        isodow(ts_local)    AS weekday,      -- 1 = Monday ... 7 = Sunday
        hour(ts_local)    AS hour,
        price_euro_mwh <0   AS is_negative
    FROM with_local
""").df()

,Count
0,70801


In [35]:
con.sql(""" 
    SELECT
        year,
        SUM(CASE WHEN is_negative THEN 1 ELSE 0 END)           AS n_negative,
        COUNT(*)                            AS n_hours,
        ROUND(100.0 * AVG(CASE WHEN is_negative THEN 1 ELSE 0 END), 1)       AS share_pct
    FROM prices_clean
    GROUP BY year
    ORDER BY year
""").df()

,year,n_negative,n_hours,share_pct
0,2018,27.0,2209,1.2
1,2019,211.0,8760,2.4
2,2020,298.0,8784,3.4
3,2021,139.0,8760,1.6
4,2022,69.0,8760,0.8
5,2023,301.0,8760,3.4
6,2024,457.0,8784,5.2
7,2025,573.0,8760,6.5
8,2026,462.0,7224,6.4


In [36]:
   con.sql("""
       SELECT hour,
              ROUND(100.0 * AVG(CASE WHEN is_negative THEN 1 ELSE 0 END), 1) AS share_pct
       FROM prices_clean
       GROUP BY hour
       ORDER BY hour
   """).df()

,hour,share_pct
0,0,1.5
1,1,2.0
2,2,2.5
3,3,2.7
4,4,2.7
5,5,2.0
6,6,1.8
7,7,1.6
8,8,1.4
9,9,2.2


## SQL first steps: findings

### Data quality checks (on prices_raw)
- 70,897 rows, 70,801 with a price. The 96 missing ones are future hours
  not yet published (as of 2026-10-28).
- No duplicate timestamps.
- Every German day has 24 hours, except clock-change days: 23 hours on the
  last Sunday of March, 25 on the last Sunday of October. No missing hours
  in 8 years.
- This also explains the extra row: 422 weeks x 168 = 70,896, plus one extra
  hour from the October 2018 clock change.

### Cleaning decisions (prices_clean)
- Removed the 96 future hours without a price.
- Kept times in UTC (ts_utc) and added German time (ts_local) for analysis.
- Added year, month, weekday (1 = Monday ... 7 = Sunday), hour and
  is_negative, all based on German time.

### Validation against pandas
- Comparing SQL with the pandas notebook revealed a bug: pandas took month,
  weekday and hour from UTC instead of German time. Fixed in notebook 02.
- After the fix, both tools give the same results.

### Negative hours by hour of day (German time)
- Peak 13:00–15:00 (about 12%), almost none 18:00–22:00 (0.1–0.4%),
  small night bump 2:00–4:00 (about 2.7%).